In [6]:
from src.MPOptoClass import *
from src.utils.gen_utils import *
from src.utils.filters import *
from src.helpers.experiment import *
from src.wiener_filter import *
from src.modeller import *

import copy
import time
import mat73 
import pynapple as nap
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.decomposition import PCA
from itertools import permutations, compress, product



%load_ext autoreload 
%autoreload 2
%matplotlib widget

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
pre=100
post=100
session_path = '/home/mirilab/diya_linux/mp_opto/data/vgat2_06062023'
session = MPOptoClass(session_path, post=post)

num_RFA_PCs = 100

(all_nlags_PCA, all_cut_PCA), (CFA_PCObj, RFA_PCObj) = session.format_nlags_PCA(bounds=session.climbing_bounds, binsize=10, nlags=10)
feature_end = (session.num_CFA*10) + (num_RFA_PCs*10)
all_nlags_PCA = all_nlags_PCA[:40000, :feature_end]
all_cut_PCA = all_cut_PCA[:40000, :num_RFA_PCs]

love


In [8]:
weights = np.logspace(0, 6, 15)
weight_combos = list(product(weights, weights))

In [9]:
psth_bounds = session.get_highlaser_bounds()
psth_ctrl_bounds = session.laser['ctrl_bounds']
psth_ctrl_bounds[:,0] = psth_ctrl_bounds[:,0] - pre
psth_bounds[:,0] = psth_bounds[:,0] - pre
psth_newbounds, climbing_duration = reBoundInBounds(session.climbing_bounds, psth_bounds)
psth_logical = bounds2Logical(psth_newbounds, duration=climbing_duration)
psth_logical_trials = unstitchSeams(psth_logical, getSeamsFromBounds(session.climbing_bounds, binsize=1))

sw_list = []
for trial in psth_logical_trials:
    temp = bin_timeseries(trial, binsize=10)
    sw_list.append(format_single_array(temp))
sw = (np.hstack(sw_list))[:-1]

sw = (sw*4)+1

In [ ]:
h_list = []
for weights in tqdm(weight_combos):
    C = np.zeros(all_nlags_PCA.shape[1] + 1)
    C[1:(session.num_CFA * 10)+1] = weights[0]
    C[(session.num_CFA * 10)+1:(feature_end+1)] = weights[1]
    print(C.shape)
    print(all_nlags_PCA.shape)
    
    h_list.append(weighted_parameter_fit(all_nlags_PCA, all_cut_PCA, c=C, sw=sw[:40000]))
    time.sleep(20)
pdump(h_list, '/home/mirilab/diya_linux/mp_opto/picklejar/h_sw_sweep')

  0%|                                                                                                                                                               | 0/225 [00:00<?, ?it/s]

(1911,)
(40000, 1910)
